# Generate platinum-arm figures

End-to-end driver that builds the four platinum-arm figures from the survival pipeline outputs:

1. **Paired volcano** (`paired_volcano_platinum.{png,pdf}`) — univariate Cox at landmarks 0d and +90d, colored by clinical category.
2. **Grouped AUC barplot** (`auc_grouped_barplot_platinum.{png,pdf}`) — test mean AUC(t) for Elastic-Net Cox, XGBoost Survival, and Dynamic DeepHit at both landmarks.
3. **2×2 importance grid** (`model_importance_platinum.{png,pdf}`) — top-15 features by Cox log-HR coefficient (top row) and XGBoost gain (bottom row) at both landmarks.
4. **PGS interpretation** (`pgs_interpretation_platinum.{png,pdf}`) — PGS-only vs lab-adjusted PGS HRs for Testosterone and PSA at both landmarks, classified into mediated / independent / suppressor / no-signal regimes.

Each figure section is independent once the shared `Imports`, `Paths`, and `Clinical category mapping` cells have been run.

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Paths

`BASE` points at the directory containing `cox/landmark_{lm}/...`, `xgboost/landmark_{lm}/...`, and `deephit/landmark_{lm}/...` outputs. Override for local copies.

In [ ]:
BASE = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/"
    "survival_analysis/PROFILE/local_runs"
)
OUT_DIR = Path("/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS/PROFILE")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS = [0, 90]
TOP_N = 15  # number of features shown per importance panel

## Clinical category mapping

40 labs in six buckets. `Body height` is dropped due to data-quality concerns. PT is folded into LFT since it tracks hepatic synthetic function. The same palette is reused across all three figures.

In [ ]:
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
ANDROGEN = {"PSA", "Testosterone"}
OTHER = {"TSH"}
DROP = {"Body height"}

CATEGORY_MAP = {}
for label, members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Vitals", VITALS), ("Androgen axis", ANDROGEN), ("Other", OTHER),
]:
    for m in members:
        CATEGORY_MAP[m] = label

DRAW_ORDER = ["Other", "Vitals", "CMP", "LFT", "CBC", "Androgen axis"]
LEGEND_ORDER = ["Androgen axis", "CBC", "LFT", "CMP", "Vitals", "Other"]

CATEGORY_COLORS = {
    "Androgen axis": "#8e1c2b",
    "CBC":           "#16a085",
    "LFT":           "#e67e22",
    "CMP":           "#7d3c98",
    "Vitals":        "#5d6d7e",
    "Other":         "#95a5a6",
}
NS_COLOR = "#d5d8dc"


def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(lab_name, "Other")


def format_label(lab_name, feature_stat):
    if not feature_stat or pd.isna(feature_stat) or feature_stat == "":
        return lab_name
    return f"{lab_name} ({feature_stat})"


def parse_feature(name):
    """Split 'LAB_NAME__stat' into (lab_name, stat). Returns (name, '') if no '__'."""
    if "__" in name:
        lab, stat = name.split("__", 1)
        return lab, stat
    return name, ""

## Figure 1 — Paired volcano (univariate Cox)

Each dot is one (lab × feature_stat) pair. ns features (q ≥ 0.05) are faded gray; q-significant features are colored by their clinical category. Three narrative beats highlighted on the figure:

- **Testosterone** is a key indicator at both landmarks
- **PSA** becomes significant at +90d
- **CBC + LFT** signals (general health markers) emerge by +90d

Source: `cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`.

In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 6
ALWAYS_LABEL = {"Hemoglobin", "Albumin", "Alkaline phosphatase"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5
MIN_LABEL_SPACING_NLP = 0.55
PANEL_XLIM = (-1.5, 1.5)
Y_MAX_CAP = 30  # -log10(p) ceiling; anything above this is drawn at the cap as a triangle


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    """Stack labels at the left/right panel edges with leader lines.

    Selection rules:
      - Androgen axis: every significant (lab × stat) is labeled (no dedup).
      - Other categories: dedupe to the most-significant stat per lab,
        then include `always_label` whitelist + top_k by p-value.
    """
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    androgen_rows = sig.loc[sig["category"] == "Androgen axis"]

    non_andro = sig.loc[sig["category"] != "Androgen axis"]
    best = (non_andro.sort_values("p_value")
                     .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = (best.loc[~best["lab_name"].isin(always_label)].head(top_k))
    non_andro_label = pd.concat([always_sig, extra]).drop_duplicates("lab_name")

    to_label = pd.concat([androgen_rows, non_andro_label]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        if side == "left":
            label_x = xlim[0] + 0.04 * xspan
            ha = "left"
        else:
            label_x = xlim[1] - 0.04 * xspan
            ha = "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = min(r["neglog10p"], Y_MAX_CAP)
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            text = label_format.format(lab=r["lab_name"],
                                       stat=r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text, xy=(r["coef_feature"], min(r["neglog10p"], Y_MAX_CAP)),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    left = to_label.loc[to_label["coef_feature"] < 0]
    right = to_label.loc[to_label["coef_feature"] >= 0]
    _stack(left, "left")
    _stack(right, "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.loc[~sub["lab_name"].isin(DROP)].copy()
    sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05
    # Cap extreme -log10(p) so a handful of huge values don't squash the rest.
    sub["capped"] = sub["neglog10p"] > Y_MAX_CAP
    sub["y"] = sub["neglog10p"].clip(upper=Y_MAX_CAP)

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["y"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        is_hero = cat == "Androgen axis"
        base_kw = dict(
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
        )
        in_range = cat_sig.loc[~cat_sig["capped"]]
        if not in_range.empty:
            ax.scatter(
                in_range["coef_feature"], in_range["y"],
                s=80 if is_hero else 40, marker="o",
                zorder=5 if is_hero else 3, **base_kw,
            )
        over = cat_sig.loc[cat_sig["capped"]]
        if not over.empty:
            ax.scatter(
                over["coef_feature"], over["y"],
                s=110 if is_hero else 60, marker="^",
                zorder=6 if is_hero else 4, **base_kw,
            )
            for _, r in over.iterrows():
                ax.annotate(
                    f"p={r['p_value']:.1e}",
                    xy=(r["coef_feature"], Y_MAX_CAP),
                    xytext=(0, -8), textcoords="offset points",
                    ha="center", va="top",
                    fontsize=7.5, color=CATEGORY_COLORS[cat],
                    zorder=7,
                )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["y"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0, color="grey", linestyle="-", linewidth=0.7, zorder=0)
    for x in (-0.5, 0.5):
        ax.axvline(x, color="grey", linestyle="--", linewidth=0.6,
                   alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    short = {"Androgen axis": "Androgen", "CBC": "CBC", "LFT": "LFT",
             "CMP": "CMP", "Vitals": "Vitals"}
    bd_str = "  ".join(f"{short[c]} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")

In [ ]:
# Load both landmarks and tag with landmark_days (authoritative from path).
def _load_uni(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
    df = pd.read_csv(p)
    df["landmark_days"] = landmark
    return df

uni = pd.concat([_load_uni(lm) for lm in LANDMARKS], ignore_index=True)
uni = uni.loc[uni["endpoint"] == "platinum"].copy()
uni = uni.dropna(subset=["coef_feature", "p_value", "q_value"])
print(f"{len(uni)} (lab × stat) rows across landmarks "
      f"{sorted(uni['landmark_days'].unique())}")

# Filter unstable Cox estimates: |log HR| > 2 or CI spans > 2 orders of magnitude.
COEF_CAP = 2.0
CI_RATIO_CAP = 100
ci_ratio = uni["ci_upper"] / uni["ci_lower"]
mask = (uni["coef_feature"].abs() <= COEF_CAP) & (ci_ratio < CI_RATIO_CAP)
print(f"dropping {int((~mask).sum())} / {len(uni)} unstable rows")
uni = uni.loc[mask].copy()
print(f"{len(uni)} rows remaining")

panels = [(0, "0 days"), (90, "+90 days")]
fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
for ax, (lm, title) in zip(axes, panels):
    sub = uni.loc[uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q ≥ 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"paired_volcano_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 2 — Discrimination vs. baseline (AUC + C-index)

Two panels — **mean IPCW AUC(t)** and **C-index** on the held-out test fold — comparing each full model against its baseline (hatched). The comparison cohort is auto-detected from the outputs:

- **PROFILE** (`stage_matched/` present): both the full labs model and the **age + stage** baseline are **retrained on the complete-case (stage-available) cohort**, so the head-to-head is on a matched population — patients with age + stage + labs.
- **CAIA** (no stage): full labs vs. **age**-only baseline on the **full cohort**.

Sources (platinum endpoint):
- **Cox** `…/cox_agg_multivariable_metrics.csv` (labs) & `…/cox_agg_baseline_metrics.csv` (baseline) → `test_mean_auc_t`, `test_c_index`
- **XGBoost** `…/landmark_xgboost_metrics.csv` (labs) & `…/landmark_xgboost_baseline_metrics.csv` (baseline) → `mean_auc_t`, `c_index`

Directory is `stage_matched/` for PROFILE, `both/` + `baseline/` for CAIA.

In [ ]:
# Detect the comparison regime from the outputs:
#   stage_matched/ present (PROFILE) -> compare labs vs. age+stage baseline on the
#       complete-case (stage-available) cohort.
#   otherwise (CAIA)                 -> compare labs vs. age baseline on the full cohort.
STAGE_MATCHED = any(
    (BASE / "cox" / f"landmark_{lm}" / "stage_matched" / "cox_agg_baseline_metrics.csv").exists()
    for lm in LANDMARKS
)
if STAGE_MATCHED:
    COX_LABS_DIR = COX_BASE_DIR = "stage_matched"
    XGB_LABS_DIR = XGB_BASE_DIR = "stage_matched"
    BASELINE_DESC = "age + stage"
    COHORT_NOTE = "stage-available (complete-case) cohort"
else:
    COX_LABS_DIR, COX_BASE_DIR = "both", "baseline"
    XGB_LABS_DIR, XGB_BASE_DIR = "both", "baseline"
    BASELINE_DESC = "age"
    COHORT_NOTE = "full cohort"


def _read_metrics(path, auc_col, cindex_col):
    """Return (mean_auc_t, c_index) for the platinum endpoint, or (nan, nan)."""
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        return (np.nan, np.nan)
    row = df.loc[df["endpoint"] == "platinum"]
    if row.empty:
        return (np.nan, np.nan)
    row = row.iloc[0]
    return (float(row[auc_col]), float(row[cindex_col]))


def cox_labs(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / COX_LABS_DIR / "cox_agg_multivariable_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


def cox_baseline(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / COX_BASE_DIR / "cox_agg_baseline_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


def xgb_labs(lm):
    return _read_metrics(BASE / "xgboost" / f"landmark_{lm}" / XGB_LABS_DIR / "landmark_xgboost_metrics.csv",
                         "mean_auc_t", "c_index")


def xgb_baseline(lm):
    return _read_metrics(BASE / "xgboost" / f"landmark_{lm}" / XGB_BASE_DIR / "landmark_xgboost_baseline_metrics.csv",
                         "mean_auc_t", "c_index")


# (label, loader, color, hatch). Baselines are the hatched, lighter twins.
SERIES = [
    ("Elastic-Net Cox",                    cox_labs,     "#4C72B0", None),
    (f"Cox baseline ({BASELINE_DESC})",    cox_baseline, "#9DB3D6", "//"),
    ("XGBoost Survival",                   xgb_labs,     "#B58900", None),
    (f"XGBoost baseline ({BASELINE_DESC})", xgb_baseline, "#E0CC8A", "//"),
]

# data[label][k] = (auc, cindex) at LANDMARKS[k]
data = {name: [loader(lm) for lm in LANDMARKS] for name, loader, _, _ in SERIES}

n_series = len(SERIES)
bar_width = 0.19
x_positions = np.arange(len(LANDMARKS))
offsets = (np.arange(n_series) - (n_series - 1) / 2) * bar_width

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, (ylabel, idx) in zip(axes, [("Test Mean AUC(t)", 0), ("Test C-index", 1)]):
    for i, (name, _loader, color, hatch) in enumerate(SERIES):
        vals = [data[name][k][idx] for k in range(len(LANDMARKS))]
        bar_x = x_positions + offsets[i]
        ax.bar(bar_x, [v if np.isfinite(v) else 0.0 for v in vals], bar_width,
               color=color, hatch=hatch, edgecolor="white", linewidth=0.8, label=name)
        for x, v in zip(bar_x, vals):
            if np.isfinite(v):
                ax.text(x, v + 0.006, f"{v:.3f}", ha="center", va="bottom",
                        fontsize=7.5, weight="semibold", color=color)
    finite = [data[n][k][idx] for n in data for k in range(len(LANDMARKS))
              if np.isfinite(data[n][k][idx])]
    ax.set_ylim(0.45, min(1.0, (max(finite) if finite else 0.7) * 1.12))
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days" for lm in LANDMARKS])
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
    ax.set_ylabel(ylabel)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)

axes[0].legend(loc="upper right", fontsize=8, ncol=1)
fig.suptitle(f"Labs vs. {BASELINE_DESC} baseline — platinum  ·  {COHORT_NOTE}",
             fontsize=13, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])

for ext in ("png", "pdf"):
    out = OUT_DIR / f"discrimination_vs_baseline_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 3 — 2×2 model importance grid

Top features by importance for Elastic-Net Cox (signed log HR coefficient, top row) and XGBoost Survival (gain, bottom row), one column per landmark. AGE is excluded so the focus is on lab-derived features.

Sources:
- **Cox**     `cox/landmark_{lm}/both/cox_agg_multivariable.csv`              → `endpoint == "platinum"`, column `coef`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_feature_importance.csv` → `endpoint == "platinum"`, column `gain`

In [ ]:
def load_cox_coefs(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[~df["is_age_covariate"].fillna(False).astype(bool)]
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def load_xgb_importance(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_feature_importance.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[df["feature"].str.lower() != "age"]
    df = df.loc[df["gain"].fillna(0) > 0]
    parsed = df["feature"].apply(lambda s: pd.Series(parse_feature(s),
                                                     index=["lab_name", "feature_stat"]))
    df = pd.concat([df.reset_index(drop=True), parsed.reset_index(drop=True)],
                   axis=1)
    return df


def render_importance_panel(ax, df, *, kind, title):
    if df.empty:
        ax.text(0.5, 0.5, "(no features to display)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    df = df.copy()
    df["category"] = df["lab_name"].map(assign_category)

    if kind == "cox":
        df = df.reindex(df["coef"].abs().sort_values(ascending=False).index).head(TOP_N)
        df = df.iloc[::-1]
        values = df["coef"].to_numpy()
        xlabel = "log HR coefficient"
    else:  # xgb
        df = df.sort_values("gain", ascending=False).head(TOP_N).iloc[::-1]
        values = df["gain"].to_numpy()
        xlabel = "XGBoost gain"

    colors = [CATEGORY_COLORS[c] for c in df["category"]]
    y = np.arange(len(df))
    ax.barh(y, values, color=colors, edgecolor="white", linewidth=0.6)

    if kind == "cox":
        ax.axvline(0, color="black", linewidth=0.5, zorder=0)

    labels = [format_label(r["lab_name"], r["feature_stat"])
              for _, r in df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, weight="bold")


fig, axes = plt.subplots(2, 2, figsize=(13, 9.5), constrained_layout=True)
MODEL_ROWS = [
    ("cox", "Elastic-Net Cox", load_cox_coefs),
    ("xgb", "XGBoost Survival", load_xgb_importance),
]

for row, (kind, model_name, loader) in enumerate(MODEL_ROWS):
    for col, lm in enumerate(LANDMARKS):
        ax = axes[row, col]
        try:
            df = loader(lm)
        except FileNotFoundError as e:
            ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="#c0392b")
            ax.set_axis_off()
            continue
        sign = "+" if lm > 0 else ""
        title = f"{model_name}  ·  {sign}{lm} days"
        render_importance_panel(ax, df, kind=kind, title=title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"model_importance_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 4 — PGS interpretation

For each PGS column, `cox_pgs_adjusted.py` fits **three** companion Cox models:

| Fit | Model | Question it answers |
|---|---|---|
| **PGS-only** | `Surv ~ AGE + PGS` | Does the PGS predict the endpoint *at all*? |
| **lab-only (baseline)** | `Surv ~ AGE + n_obs + LAB` | What does the lab marker do on its own? |
| **joint** | `Surv ~ AGE + PGS + n_obs + LAB` | What's left of each after adjusting for the other? |

This panel pairs the **PGS-only HR** (open circle) with the **joint PGS HR** (filled diamond, colored by regime) for each PGS at each (target lab × landmark). The thin connector shows the direction and magnitude of the attenuation when the lab is added to the model.

**Four regimes** (BH q < 0.05 on each side, computed per endpoint):

- 🟢 **Independent** — PGS-only sig **and** joint sig → PGS carries information beyond the lab marker.
- 🟠 **Mediated** — PGS-only sig **but** joint not → the lab absorbs the PGS signal; PGS adds nothing on top.
- 🟣 **Suppressor** — PGS-only **not** sig but joint **is** → rare; usually collinearity / variance suppression.
- ⚪ **No signal** — neither significant → PGS is null for this endpoint × lab combination.

Source: `cox/landmark_{lm}/both/cox_pgs_adjusted.csv` (one row per `endpoint × lab × PGS`). Endpoint is fixed to platinum to match the rest of this notebook.

In [ ]:
def load_pgs(landmark, endpoint="platinum"):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_pgs_adjusted.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == endpoint].copy()
    df["landmark_days"] = landmark
    return df


REGIME_COLORS = {
    "independent": "#1abc9c",
    "mediated":    "#e67e22",
    "suppressor":  "#7d3c98",
    "no_signal":   "#bdc3c7",
}
REGIME_ORDER = ["independent", "mediated", "suppressor", "no_signal"]


def classify_regime(row, alpha=0.05):
    q_only = row.get("q_value_pgs_only", np.nan)
    q_joint = row.get("q_value_pgs", np.nan)
    if not np.isfinite(q_only) or not np.isfinite(q_joint):
        return "no_signal"
    only_sig = q_only < alpha
    joint_sig = q_joint < alpha
    if only_sig and joint_sig:
        return "independent"
    if only_sig and not joint_sig:
        return "mediated"
    if not only_sig and joint_sig:
        return "suppressor"
    return "no_signal"


def _regime_color(g):
    if not isinstance(g, str):
        return "#bdc3c7"
    return REGIME_COLORS.get(g, "#bdc3c7")


PGS_CATEGORIES = [
    ("Testosterone PGS", "testosterone"),
    ("Prostate-cancer PGS", "prostate"),
]
# Per-category figure-height multiplier. Categories with many PGSs need
# more vertical space so y-tick labels don't overlap.
HEIGHT_MULT_PER_CATEGORY = {
    "Prostate-cancer PGS": 2.0,
}


def assign_pgs_category(pgs_name: str) -> str:
    name = str(pgs_name).lower()
    for label, token in PGS_CATEGORIES:
        if token in name:
            return label
    return "Other PGS"


try:
    pgs_df = pd.concat([load_pgs(lm) for lm in LANDMARKS], ignore_index=True)
except FileNotFoundError as e:
    print(f"cox_pgs_adjusted.csv missing — skipping Figure 4 (file: {e.filename})")
else:
    pgs_df["regime"] = pgs_df.apply(classify_regime, axis=1)
    pgs_df["pgs_category"] = pgs_df["pgs"].map(assign_pgs_category)

    target_labs = ["Testosterone", "PSA"]

    for cat_label, _token in PGS_CATEGORIES:
        cat_df = pgs_df.loc[pgs_df["pgs_category"] == cat_label].copy()
        if cat_df.empty:
            print(f"[skip] {cat_label}: no rows after filtering")
            continue

        # PGS order within this category: by |coef_pgs_only|, ascending so the
        # strongest score sits at the top of the y-axis.
        ordering = (cat_df.assign(abs_pgs=cat_df["coef_pgs_only"].abs())
                          .groupby("pgs")["abs_pgs"].max()
                          .sort_values(ascending=True))
        pgs_order = list(ordering.index)

        fig, axes = plt.subplots(
            len(target_labs), len(LANDMARKS),
            figsize=(13, (4.5 * len(target_labs) + 1) * HEIGHT_MULT_PER_CATEGORY.get(cat_label, 1.0)),
            sharex=True, sharey=True,
        )
        if len(target_labs) == 1:
            axes = np.array([axes])

        for r, lab in enumerate(target_labs):
            for c, lm in enumerate(LANDMARKS):
                ax = axes[r, c]
                sub = cat_df.loc[
                    (cat_df["lab_name"] == lab) & (cat_df["landmark_days"] == lm)
                ].copy()
                if sub.empty:
                    ax.text(0.5, 0.5, "(no rows)", ha="center", va="center",
                            transform=ax.transAxes, color="#7f8c8d")
                    ax.set_axis_off()
                    continue

                sub = sub.set_index("pgs").reindex(pgs_order).reset_index()
                y_pos = np.arange(len(sub))

                only_hr = sub["hazard_ratio_pgs_only_per_sd"].to_numpy(dtype=float)
                ci_lo = sub["ci_lower_pgs_only"].to_numpy(dtype=float)
                ci_hi = sub["ci_upper_pgs_only"].to_numpy(dtype=float)
                q_only = pd.to_numeric(
                    sub.get("q_value_pgs_only", pd.Series(np.nan, index=sub.index)),
                    errors="coerce",
                ).to_numpy(dtype=float)
                sig = np.isfinite(q_only) & (q_only < 0.05)

                ax.errorbar(
                    only_hr, y_pos,
                    xerr=[np.clip(only_hr - ci_lo, 0, None),
                          np.clip(ci_hi - only_hr, 0, None)],
                    fmt="none",
                    ecolor="#5d6d7e",
                    elinewidth=1.0, capsize=0,
                    zorder=2,
                )
                # filled circle = q<0.05; open circle = ns
                if sig.any():
                    ax.scatter(only_hr[sig], y_pos[sig],
                               s=55, marker="o",
                               facecolor="#2c3e50", edgecolor="white",
                               linewidth=0.7, zorder=4)
                if (~sig).any():
                    ax.scatter(only_hr[~sig], y_pos[~sig],
                               s=45, marker="o",
                               facecolor="white", edgecolor="#5d6d7e",
                               linewidth=1.0, zorder=4)

                ax.axvline(1.0, color="grey", linestyle=":", linewidth=0.9)
                ax.set_xscale("log")
                ax.set_yticks(y_pos)
                ax.set_yticklabels(sub["pgs"], fontsize=8)
                sign = "+" if lm > 0 else ""
                ax.set_title(f"{lab}  ·  {sign}{lm} days", fontsize=11, weight="bold")
                if r == len(target_labs) - 1:
                    ax.set_xlabel("HR per SD (log scale)")

        legend_handles = [
            Line2D([0], [0], marker="o", color="w",
                   markerfacecolor="#2c3e50", markeredgecolor="white",
                   markersize=9, linewidth=0,
                   label="PGS-only HR (q<0.05)"),
            Line2D([0], [0], marker="o", color="w",
                   markerfacecolor="white", markeredgecolor="#5d6d7e",
                   markersize=8, linewidth=0,
                   label="PGS-only HR (ns)"),
        ]
        fig.legend(handles=legend_handles, loc="lower center",
                   ncol=len(legend_handles), bbox_to_anchor=(0.5, -0.02),
                   fontsize=9)
        fig.suptitle(f"PGS interpretation ({cat_label}) — platinum endpoint",
                     fontsize=13, weight="bold", y=1.0)
        fig.tight_layout(rect=[0, 0.03, 1, 0.98])

        safe = cat_label.lower().replace(" ", "_").replace("-", "_")
        for ext in ("png", "pdf"):
            out = OUT_DIR / f"pgs_interpretation_platinum_{safe}.{ext}"
            fig.savefig(out)
            print(f"wrote {out}")
        plt.show()

    print("\nRegime counts by (target lab × landmark × pgs_category):")
    summary = (
        pgs_df.groupby(["pgs_category", "lab_name", "landmark_days", "regime"])
              .size()
              .unstack("regime", fill_value=0)
              .reindex(columns=REGIME_ORDER, fill_value=0)
    )
    print(summary)

## Figure 5 — Cancer stage effect (age-adjusted baseline Cox)

Log hazard ratios for cancer stage (vs. **Stage I** reference) from the **baseline** Cox model `Surv ~ age + CANCER_STAGE_*`, grouped by landmark. Each bar's annotation is the HR (`exp(coef)`). Stage is derived from clinical text and is **PROFILE-only**; the CAIA baseline is age-only, so this panel auto-skips when no `CANCER_STAGE_*` coefficients are present.

Source: `cox/landmark_{lm}/baseline/cox_agg_baseline.csv` → `endpoint == "platinum"`, rows where `feature` starts with `CANCER_STAGE_`.

In [ ]:
STAGE_LABELS = {
    "CANCER_STAGE_II":  "Stage II",
    "CANCER_STAGE_III": "Stage III",
    "CANCER_STAGE_IV":  "Stage IV",
}
STAGE_ORDER = ["CANCER_STAGE_II", "CANCER_STAGE_III", "CANCER_STAGE_IV"]
LM_COLORS = {0: "#4C72B0", 90: "#55A868"}


def _cox_baseline_dir(landmark):
    # Prefer the complete-case (stage_matched) baseline when present; else the
    # full-cohort baseline.
    matched = BASE / "cox" / f"landmark_{landmark}" / "stage_matched" / "cox_agg_baseline.csv"
    return "stage_matched" if matched.exists() else "baseline"


def load_cox_baseline_stage(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / _cox_baseline_dir(landmark) / "cox_agg_baseline.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[df["feature"].astype(str).str.startswith("CANCER_STAGE_")]
    return df


# Collect the stage log-HRs per landmark. CAIA's baseline is age-only, so this
# stays empty there and the panel auto-skips.
stage_by_lm = {}
for lm in LANDMARKS:
    try:
        s = load_cox_baseline_stage(lm)
    except FileNotFoundError:
        s = pd.DataFrame(columns=["feature", "coef", "exp(coef)"])
    if not s.empty:
        stage_by_lm[lm] = s.set_index("feature")

if not stage_by_lm:
    print("[skip] Cancer-stage panel: no CANCER_STAGE_* coefficients "
          "(age-only baseline / no stage source for this cohort).")
else:
    lms = [lm for lm in LANDMARKS if lm in stage_by_lm]
    x = np.arange(len(STAGE_ORDER))
    n = len(lms)
    bw = 0.8 / n
    offs = (np.arange(n) - (n - 1) / 2) * bw

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for j, lm in enumerate(lms):
        s = stage_by_lm[lm]
        coefs = [float(s.loc[f, "coef"]) if f in s.index else np.nan for f in STAGE_ORDER]
        hrs = [float(s.loc[f, "exp(coef)"]) if f in s.index else np.nan for f in STAGE_ORDER]
        bx = x + offs[j]
        sign = "+" if lm > 0 else ""
        ax.bar(bx, [c if np.isfinite(c) else 0.0 for c in coefs], bw,
               color=LM_COLORS.get(lm, "#777777"), edgecolor="white",
               linewidth=0.8, label=f"{sign}{lm} days")
        for xx, c, hr in zip(bx, coefs, hrs):
            if np.isfinite(c):
                ax.text(xx, c + (0.02 if c >= 0 else -0.02),
                        f"HR {hr:.2f}",
                        ha="center", va="bottom" if c >= 0 else "top",
                        fontsize=8, color=LM_COLORS.get(lm, "#777777"))

    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([STAGE_LABELS[f] for f in STAGE_ORDER])
    ax.set_ylabel("log HR vs. Stage I (baseline Cox)")
    ax.set_title("Cancer stage effect on platinum risk — age-adjusted baseline "
                 "(complete-case cohort)", fontsize=12, weight="bold")
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    ax.legend(title="Landmark", fontsize=9)

    for ext in ("png", "pdf"):
        out = OUT_DIR / f"cancer_stage_effect_platinum.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
    plt.show()